In [ ]:
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from matplotlib.figure import figaspect

params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'font.weight': 'normal',
         'lines.markersize' : 10}
pylab.rcParams.update(params)

In [ ]:
from string import Template

master_dict={}
master_dict['what_am_I_looking_at'] = ('ligands_bending',)
master_dict['which_sim'] = ('sim_1','sim_2','sim_3','sim_4','sim_5','sim_6','sim_7', 'sim_8')
master_dict['which_quartet_number'] = ('6120',)
master_dict['which_number_per_quartet'] = ('25',)
master_dict['which_crowder'] = ('4080', )
master_dict['which_concentration'] = ('0.6',)
master_dict['which_vdW'] = ('5.0',)
master_dict['which_vdW_ligand'] = ('5.0','10.0','15.0','20.0')
master_dict['what_monomer_number'] = ('15',)
master_dict['what_monomer_number_ligand'] = ('1','3','5')

template_hndl=Template(
    "$what_am_I_looking_at/$which_sim/custom_data_wip_${which_quartet_number}_${which_number_per_quartet}_${which_crowder}_${which_concentration}_${which_vdW}_${which_vdW_ligand}_${what_monomer_number}_${what_monomer_number_ligand}.h5"
    )

In [ ]:
from collections import defaultdict
import gzip
import pickle

def open_data(processed_data_path, key=''):
    f = gzip.open(processed_data_path, 'rb')
    pool_assignments = pickle.load(f)
    f.close()
    pool_data = pool_assignments[key]
    return pool_data.keys(), list(pool_data.values())


def get_transparency_loc(val,alphas):
    if val==1:
        return 0
    if val==3:
        return 1
    if val==5:
        return 2
    else:
        raise ValueError

In [ ]:
import numpy as np
from pmtools.runner import Engine
from pmtools.kernels import calculate_stacking_fraction

master_dict['what_am_I_looking_at'] = ('ligands_nobending',)
master_dict['what_monomer_number_ligand'] = ('1','3',)

kernel_kwargs={'particle_group':'Filament','particle_group_alt':'Crowder','chunk': (0, None, 1), 'box_dim': np.array([445.15987681038695]*3), 'crit': 1.3, 'extra_flag': 'concentration_variation'}
engine_inst=Engine(world_path='{path_to_data}/')
engine_inst.assemble_paths(master_dict,template_hndl,'which_sim')
engine_inst.register_kernel(calculate_stacking_fraction, **kernel_kwargs)
engine_inst.run(max_workers=12)
engine_inst.collect_results()
engine_inst.save_results(filename='g4polyligand_stacking_fraction.p.gz', custom_full_path='{path_to_data}/test_local_analysis/g4polyligand_stacking_fraction_soft.p.gz')
engine_inst.shutdown()
print('DONE!')

In [ ]:
import numpy as np
from pmtools.runner import Engine
from pmtools.kernels import calculate_stacking_fraction

master_dict['what_am_I_looking_at'] = ('ligands_bending',)
master_dict['what_monomer_number_ligand'] = ('1', '3', '5')

kernel_kwargs={'particle_group':'Filament','particle_group_alt':'Crowder','chunk': (0, None, 1), 'box_dim': np.array([445.15987681038695]*3), 'crit': 1.3, 'extra_flag': 'concentration_variation'}
engine_inst=Engine(world_path='{path_to_data}/')
engine_inst.assemble_paths(master_dict,template_hndl,'which_sim')
engine_inst.register_kernel(calculate_stacking_fraction, **kernel_kwargs)
engine_inst.run(max_workers=12)
engine_inst.collect_results()
engine_inst.save_results(filename='g4polyligand_stacking_fraction.p.gz', custom_full_path='{path_to_data}/test_local_analysis/g4polyligand_stacking_fraction.p.gz')
engine_inst.shutdown()
print('DONE!')

In [ ]:
from pmtools.refractored_toolbox import determine_key_val_from_filename

markers = ['s', 'o','*']
linestyles = ['--', 'dotted']
notes = ['10', '15', '20']
clr = plt.cm.tab10(np.linspace(0, 1, 4, endpoint=True))
alphas=np.linspace(0.25,1,4)

processed_data_paths = '{path_to_data}/test_local_analysis/g4polyligand_stacking_fraction.p.gz'

labels=['monoligand','polyligand (x3)','polyligand (x5)']
titles=[r'$\epsilon=5$',r'$\epsilon=10$',r'$\epsilon=15$',r'$\epsilon=20$']

w, h = figaspect(0.3)
vals=[0,1,2,3]
f, axs = plt.subplots(1,len(vals),sharey=True,sharex=True,figsize=(w,h))

pool_lbl, pool_data = open_data(processed_data_paths, key='calculate_stacking_fraction')
data_dict=defaultdict(list)
for lbl,data_el in zip(pool_lbl,pool_data):
    stuff=determine_key_val_from_filename(template_hndl,lbl,'which_vdW_ligand')
    other_stuff=determine_key_val_from_filename(template_hndl,lbl,'what_monomer_number_ligand')
    for data_el_el in data_el:
        yax=[]
        for data_xxx in list(data_el_el.values())[0]:
            res=len([x for x in data_xxx.values() if len(x)==2])
            yax.append(res)
        data_dict[f'{stuff}_{other_stuff}'].append(yax)

for lbl,data_el in data_dict.items():
    stuff=float(lbl.split('_')[0])
    if stuff==5:
        
        mover=0
    elif stuff==10:
        
        mover=1
    elif stuff==15:
        
        mover=2
    else:
        mover=3
    
    mark_lbl_index=get_transparency_loc(float(lbl.split('_')[-1]),alphas)
    y_mean = np.mean(data_el, axis=0)
    y_std = np.std(data_el, axis=0)
    axis = np.arange(len(y_mean)) * 2.5 + 12.5
    axs[mover].errorbar(axis,y_mean/4080.,yerr=y_std/4080.,color=clr[mover],marker=markers[mark_lbl_index], label=labels[mark_lbl_index],markerfacecolor='none')

labels=['soft monoligand','soft polyligand (x3)','polyligand (x5)']
processed_data_paths = '{path_to_data}/test_local_analysis/g4polyligand_stacking_fraction_soft.p.gz'
pool_lbl, pool_data = open_data(processed_data_paths, key='calculate_stacking_fraction')

data_dict=defaultdict(list)
for lbl,data_el in zip(pool_lbl,pool_data):
    stuff=determine_key_val_from_filename(template_hndl,lbl,'which_vdW_ligand')
    other_stuff=determine_key_val_from_filename(template_hndl,lbl,'what_monomer_number_ligand')
    for data_el_el in data_el:
        yax=[]
        for data_xxx in list(data_el_el.values())[0]:
            res=len([x for x in data_xxx.values() if len(x)==2])
            yax.append(res)
        data_dict[f'{stuff}_{other_stuff}'].append(yax)

axis=np.array([x*2.5 for x in range(16)])
for lbl,data_el in data_dict.items():
    stuff=float(lbl.split('_')[0])
    if stuff==5:
        
        continue
    elif stuff==10:
        
        mover=1
    elif stuff==15:
        
        mover=2
    else:
        continue
    
    mark_lbl_index=get_transparency_loc(float(lbl.split('_')[-1]),alphas)
    if mark_lbl_index==1:
        axs[mover].errorbar(axis[5:],np.mean(data_el,axis=0)[5:]/4080.,yerr=np.std(data_el,axis=0)[5:]/4080.,color=clr[mover],marker=markers[mark_lbl_index], label=labels[mark_lbl_index])
    
for zvrk,ax in enumerate(axs):
    handles, labels = ax.get_legend_handles_labels()
    # make a dict which will keep only the first handle per label
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(),labelspacing=0.1, columnspacing=0.01, handletextpad=0.04, borderpad=0.1)
    ax.set_ylim(0,0.75)
    ax.set_title(titles[zvrk])
    ax.grid(True)
f.supxlabel(r'Simulation time [$\mu s$]',fontsize='x-large')
axs[0].set_ylabel('Fraction',fontsize='x-large')

In [ ]:
from pmtools.runner import Engine
from pmtools.kernels import per_fil_gyr

def exclude_patches(subset):
    return subset.type != 4

def is_g4_filament(subset):
    return not np.any(subset.type == 5)

kernel_kwargs={'particle_group':'Filament','particle_group_alt':'Crowder','chunk': (-5, None, 1), 'box_dim': np.array([445.15987681038695]*3), 'crit': 1.3, 'extra_flag': 'concentration_variation', 'particle_predicate': exclude_patches,'object_predicate': is_g4_filament}
engine_inst=Engine(world_path='{path_to_data}/')
engine_inst.assemble_paths(master_dict,template_hndl,'which_sim')
engine_inst.register_kernel(per_fil_gyr, **kernel_kwargs)
engine_inst.run(max_workers=12)
engine_inst.collect_results()
engine_inst.save_results(filename='g4polyligand_gyration.p.gz', custom_full_path='{path_to_data}/test_local_analysis/g4polyligand_gyration.p.gz')
engine_inst.shutdown()
print('DONE!')

In [ ]:
from pmtools.refractored_toolbox import determine_key_val_from_filename

markers = ['s', 'o','*']
linestyles = ['--', 'dotted']
notes = ['10', '15', '20']
clr = plt.cm.tab10(np.linspace(0, 1, 4, endpoint=True))
alphas=np.linspace(0.25,1,4)

processed_data_paths = '{path_to_data}/test_local_analysis/g4polyligand_gyration.p.gz'

pool_lbl, pool_data = open_data(processed_data_paths, key='per_fil_gyr')

data_dict = defaultdict(list)

for lbl,data_el in zip(pool_lbl,pool_data):
    stuff=determine_key_val_from_filename(template_hndl,lbl,'which_vdW_ligand')
    other_stuff=determine_key_val_from_filename(template_hndl,lbl,'what_monomer_number_ligand')
    for data_el_el in data_el:
        yax=[]
        for data_xxx in list(data_el_el.values())[0]:
            yax.append(np.sqrt(data_xxx.get_R2())/26.051226)
            
        data_dict[f'{stuff}_{other_stuff}'].extend(yax)

x_vals = []
y_vals = []
z_vals = []

for key, z_list in data_dict.items():
    x_str, y_str = key.split("_")
    x_val = float(x_str)
    y_val = float(y_str)

    z_avg = np.mean(z_list)
    z_vals.append(z_avg)
    x_vals.append(x_val)
    y_vals.append(y_val)

x_vals = np.array(x_vals)
y_vals = np.array(y_vals)
z_vals = np.array(z_vals) 

Z_target = z_vals

xi = np.unique(x_vals)
yi = np.unique(y_vals)
Xi, Yi = np.meshgrid(xi, yi)

Zi = np.empty_like(Xi)
for i in range(len(yi)):
    for j in range(len(xi)):
        mask = (x_vals == xi[j]) & (y_vals == yi[i])
        Zi[i, j] = Z_target[mask][0] if np.any(mask) else np.nan

# Plot contour
plt.figure(figsize=(6, 5))
cp = plt.contourf(Xi, Yi, Zi, cmap='viridis')
plt.colorbar(cp)
plt.xlabel(r"$\epsilon$")
plt.ylabel(r"$M$")
plt.title("Gyration Radius")
plt.grid(linestyle='dotted')


In [ ]:
markers = ['s', 'o','*']
linestyles = ['--', 'dotted']
notes = ['10', '15', '20']
clr = plt.cm.tab10(np.linspace(0, 1, 4, endpoint=True))
alphas=np.linspace(0.25,1,4)

processed_data_paths = '{path_to_data}/test_local_analysis/g4polyligand_gyration.p.gz'

pool_lbl, pool_data = open_data(processed_data_paths, key='per_fil_gyr')

data_dict = defaultdict(list)

for lbl,data_el in zip(pool_lbl,pool_data):
    stuff=determine_key_val_from_filename(template_hndl,lbl,'which_vdW_ligand')
    other_stuff=determine_key_val_from_filename(template_hndl,lbl,'what_monomer_number_ligand')
    for data_el_el in data_el:
        yax=[]
        for data_xxx in list(data_el_el.values())[0]:
            x,y,z=data_xxx.eigenvalues
            yax.append(3/2*(x**2+y**2+z**2)/(x+y+z)**2-1/2)
        data_dict[f'{stuff}_{other_stuff}'].extend(yax)

x_vals = []
y_vals = []
z_vals = []

for key, z_list in data_dict.items():
    x_str, y_str = key.split("_")
    x_val = float(x_str)
    y_val = float(y_str)

    z_avg = np.mean(z_list) 
    z_vals.append(z_avg)
    x_vals.append(x_val)
    y_vals.append(y_val)

x_vals = np.array(x_vals)
y_vals = np.array(y_vals)
z_vals = np.array(z_vals)  

Z_target = z_vals

xi = np.unique(x_vals)
yi = np.unique(y_vals)
Xi, Yi = np.meshgrid(xi, yi)

Zi = np.empty_like(Xi)
for i in range(len(yi)):
    for j in range(len(xi)):
        mask = (x_vals == xi[j]) & (y_vals == yi[i])
        Zi[i, j] = Z_target[mask][0] if np.any(mask) else np.nan

plt.figure(figsize=(6, 5))
cp = plt.contourf(Xi, Yi, Zi,cmap='cividis')
plt.colorbar(cp)
plt.xlabel(r"$\epsilon$")
plt.ylabel(r"$M$")
plt.grid(linestyle='dotted')
plt.title("Relative Shape Anisotropy")


In [ ]:
from pmtools.runner import Engine
from pmtools.kernels import lp_projection

def exclude_patches(subset):
    return subset.type != 4

def is_g4_filament(subset):
    return not np.any(subset.type == 5)

master_dict['what_am_I_looking_at'] = ('ligands_nobending',)
master_dict['what_monomer_number_ligand'] = ('1','3',)

kernel_kwargs={'particle_group':'Filament','particle_group_alt':'Crowder','chunk': (-5, None, 1), 'box_dim': np.array([445.15987681038695]*3), 'crit': 1.3, 'extra_flag': 'concentration_variation', 'particle_predicate': exclude_patches,'object_predicate': is_g4_filament}
engine_inst=Engine(world_path='{path_to_data}/')
engine_inst.assemble_paths(master_dict,template_hndl,'which_sim')
engine_inst.register_kernel(lp_projection, **kernel_kwargs)
engine_inst.run(max_workers=12)
engine_inst.collect_results()
engine_inst.save_results(filename='g4polyligand_lp_soft.p.gz', custom_full_path='{path_to_data}/test_local_analysis/g4polyligand_lp_soft.p.gz')
engine_inst.shutdown()
print('DONE!')

In [ ]:
from pmtools.runner import Engine
from pmtools.kernels import lp_projection

def exclude_patches(subset):
    return subset.type != 4

def is_g4_filament(subset):
    return not np.any(subset.type == 5)

master_dict['what_am_I_looking_at'] = ('ligands_bending',)
master_dict['what_monomer_number_ligand'] = ('1','3','5')

kernel_kwargs={'particle_group':'Filament','particle_group_alt':'Crowder','chunk': (-5, None, 1), 'box_dim': np.array([445.15987681038695]*3), 'crit': 1.3, 'extra_flag': 'concentration_variation', 'particle_predicate': exclude_patches,'object_predicate': is_g4_filament}
engine_inst=Engine(world_path='{path_to_data}/')
engine_inst.assemble_paths(master_dict,template_hndl,'which_sim')
engine_inst.register_kernel(lp_projection, **kernel_kwargs)
engine_inst.run(max_workers=12)
engine_inst.collect_results()
engine_inst.save_results(filename='g4polyligand_lp_soft.p.gz', custom_full_path='{path_to_data}/test_local_analysis/g4polyligand_lp.p.gz')
engine_inst.shutdown()
print('DONE!')

In [ ]:
from scipy.optimize import curve_fit

def process_Lp_modern(pool_lbl, pool_data, selection=None):
    for iid, (lbl, elem) in enumerate(zip(pool_lbl, pool_data)):
        if selection == lbl or selection == None :
            data = []
            for sim in elem:
                try:
                    popt, bond_l = list(sim.values())[0]
                    data.append(popt)
                except TypeError:
                    print('havarija')
                    continue
            data = np.array(data)
            if len(data) != 8:
                print(f"WARNING: expected 8 simulations for {lbl}, got {len(data)}")
            data_plt = np.mean(data, axis=0)
            data_std = np.std(data, axis=0)
            yield lbl, bond_l, data_plt, data_std, len(data)
            
def stranger_things(k, alpha, nu):
    global hlt
    return alpha*pow(k*(hlt-k)/hlt, 2*nu-1)


markers = ['s', 'o','*']
errorbar_style = dict(markersize=plt.rcParams['lines.markersize'], elinewidth=1.0, capsize=2, capthick=1.0)
linestyles = ['--', 'dotted']
notes = ['10', '15', '20']
clr = plt.cm.tab10(np.linspace(0, 1, 4, endpoint=True))
alphas=np.linspace(0.25,1,4)
labels=['monoligand','polyligand (x3)','polyligand (x5)']

processed_data_paths = '{path_to_data}/test_local_analysis/g4polyligand_lp.p.gz'

grouped_data = defaultdict(list)
pool_lbl, pool_data = open_data(processed_data_paths, key='lp_projection')
process_gen = process_Lp_modern(pool_lbl, pool_data)
while True:
    try:
        lbl, axisa, scatter_data, scatter_std, sim_count = next(process_gen)
        param_val = determine_key_val_from_filename(template_hndl, lbl, 'which_vdW_ligand')
        grouped_data[param_val].append((lbl, axisa, scatter_data, scatter_std, sim_count))
    except StopIteration:
        break

unique_values = sorted(grouped_data.keys())

w, h = figaspect(0.3)
f, axs = plt.subplots(1, len(unique_values), sharey=True, sharex=True, figsize=(w, h))
if len(unique_values) == 1:
    axs = [axs]

for ax, param in zip(axs, unique_values):
    curves = grouped_data[param]
    for lbl, axisa, scatter_data, scatter_std, sim_count in curves:
        stuff=int(determine_key_val_from_filename(template_hndl,lbl,'which_vdW_ligand'))
        if stuff==5:
            mover=0
        elif stuff==10:
            mover=1
        elif stuff==15:
            mover=2
        else:
            mover=3
        lbl_flt = lbl.split('/')[-1][37:]
        hlt = max(axisa)
        popt, pcov = curve_fit(stranger_things, axisa[:-1], scatter_data[:-1])

        # Normalize the x-data and scale the y-data as in your original approach.
        norm_axisa = np.array(axisa) / max(axisa)
        scaled_scatter = np.array(scatter_data)
        scatter_std = np.array(scatter_std)
        mark_lbl_index=get_transparency_loc(determine_key_val_from_filename(template_hndl,lbl,'what_monomer_number_ligand'),alphas)
        ax.errorbar(norm_axisa, scaled_scatter, yerr=scatter_std, color=clr[mover], marker=markers[mark_lbl_index], markerfacecolor='none', label=labels[mark_lbl_index], linestyle='none', **errorbar_style)
        ax.plot(axisa[:-1]/max(axisa)+0.03, stranger_things(axisa[:-1],*popt),color=clr[mover])
    ax.set_title(f"$\epsilon$ = {int(param)}")
    ax.legend(labelspacing=0.1, columnspacing=0.01, handletextpad=0.04, borderpad=0.1)
f.supxlabel(r'$k/(M-1)$',fontsize='x-large')
axs[0].set_ylabel(r'$L_p(k)$',fontsize='x-large')

processed_data_paths = '{path_to_data}/test_local_analysis/g4polyligand_lp_soft.p.gz'
labels=['soft monoligand','soft polyligand (x3)','polyligand (x5)']

grouped_data = defaultdict(list)
pool_lbl, pool_data = open_data(processed_data_paths, key='lp_projection')
process_gen = process_Lp_modern(pool_lbl, pool_data)
while True:
    try:
        lbl, axisa, scatter_data, scatter_std, sim_count = next(process_gen)
        param_val = determine_key_val_from_filename(template_hndl, lbl, 'which_vdW_ligand')
        grouped_data[param_val].append((lbl, axisa, scatter_data, scatter_std, sim_count))
    except StopIteration:
        break

unique_values = sorted(grouped_data.keys())
for ax, param in zip(axs, unique_values):
    curves = grouped_data[param]
    for lbl, axisa, scatter_data, scatter_std, sim_count in curves:
        stuff=int(determine_key_val_from_filename(template_hndl,lbl,'which_vdW_ligand'))
        if stuff==5:
            continue
        elif stuff==10:
            mover=1
        elif stuff==15:
            continue
        else:
            continue
        lbl_flt = lbl.split('/')[-1][37:]
        hlt = max(axisa)
        popt, pcov = curve_fit(stranger_things, axisa[:-1], scatter_data[:-1])

        norm_axisa = np.array(axisa) / max(axisa)
        scaled_scatter = np.array(scatter_data)
        scatter_std = np.array(scatter_std)
        mark_lbl_index=get_transparency_loc(determine_key_val_from_filename(template_hndl,lbl,'what_monomer_number_ligand'),alphas)
        if mark_lbl_index==1:
            ax.errorbar(norm_axisa, scaled_scatter, yerr=scatter_std, color=clr[mover], marker=markers[mark_lbl_index], label=labels[mark_lbl_index], linestyle='none', **errorbar_style)
            ax.plot(axisa[:-1]/max(axisa)+0.03, stranger_things(axisa[:-1],*popt),color=clr[mover])
    ax.set_title(f"$\epsilon$ = {int(param)}")
    ax.legend(labelspacing=0.1, columnspacing=0.01, handletextpad=0.04, borderpad=0.1)
f.supxlabel(r'$k/(M-1)$',fontsize='x-large')
axs[0].set_ylabel(r'$L_p(k)$',fontsize='x-large')

In [ ]:
from pmtools.runner import Engine
import numpy as np
import pmtools.refractored_toolbox as context
from pmtools.resources.kernel_config import AnalysisConfig
from pressomancy.analysis import H5DataSelector
from pressomancy.helper_functions import get_neighbours_cross_lattice
import h5py

def calculate_ligand_cooperativity(cfg: AnalysisConfig):
    """
    Measure per-polyligand intercalation occupancy and cooperativity.

    This uses the same monomer-level intercalation definition as
    :func:`calculate_stacking_fraction`: a ligand monomer is intercalated when
    it has exactly two neighbouring stacking sites within ``cfg.crit``. The
    monomer hits are then grouped by parent ligand object using the
    pressomancy connectivity API.

    Parameters
    ----------
    cfg : AnalysisConfig
        Requires ``data_path``, ``particle_group`` (stacking-site group),
        ``particle_group_alt`` (ligand-monomer group), ``box_dim``, ``crit``,
        ``chunk``, and ``template_hndl`` when ligand length can be inferred
        from the filename.

    Returns
    -------
    dict
        ``{cfg.data_path: result}``, where ``result`` contains timestep-aligned
        arrays for ``occupancy_counts``, ``occupancy_prob``, ``p_int``, and
        ``C``. ``C`` is ``nan`` for monoligands or when ``p_int == 0``.
    """

    data_with_context = {}
    data_file = h5py.File(cfg.data_path, "r")
    data = H5DataSelector(data_file, particle_group=cfg.particle_group)
    data_other = H5DataSelector(data_file, particle_group=cfg.particle_group_alt)

    try:
        ligand_length = int(context.determine_key_val_from_filename(
            cfg.template_hndl, cfg.data_path, 'what_monomer_number_ligand'))
    except Exception:
        ligand_length = None

    crowder_ids = np.asarray(data_other.get_connectivity_values(cfg.particle_group_alt), dtype=int)
    filament_to_crowder = None
    if ligand_length != 1:
        filament_to_crowder = data.get_connectivity_map("Filament", "Crowder")

    if filament_to_crowder is None or ligand_length == 1:
        ligand_length = 1
        parent_to_children = {int(crowder_id): [int(crowder_id)] for crowder_id in crowder_ids}
        child_to_parent = {int(crowder_id): int(crowder_id) for crowder_id in crowder_ids}
    else:
        filament_to_crowder = np.asarray(filament_to_crowder, dtype=int)
        crowder_id_set = set(int(crowder_id) for crowder_id in crowder_ids)
        parent_to_children = {}
        child_to_parent = {}
        for parent_id, child_id in filament_to_crowder:
            parent_id = int(parent_id)
            child_id = int(child_id)
            if child_id not in crowder_id_set:
                continue
            parent_to_children.setdefault(parent_id, []).append(child_id)
            child_to_parent[child_id] = parent_id

        if ligand_length is None:
            child_counts = [len(children) for children in parent_to_children.values()]
            ligand_length = max(child_counts) if child_counts else 1

    parent_ids = np.array(sorted(parent_to_children), dtype=int)
    n_ligands = len(parent_ids)
    start, end, step = cfg.chunk

    g4_multimer_ids = data.timestep[0].get_connectivity_values(
        "Filament", predicate=lambda subset: np.any(subset.type == 4))
    stacking_site_to_multimer = {}
    for multimer_id in g4_multimer_ids:
        stacking_subset = data.timestep[0].select_particles_by_object(
            "Filament", multimer_id, predicate=lambda subset: subset.type == 4)
        for particle_id in np.ravel(stacking_subset.id).astype(int):
            stacking_site_to_multimer[int(particle_id)] = int(multimer_id)

    max_bridge_count = 2 * ligand_length
    occupancy_counts = []
    bridge_counts = []
    joint_occupancy_bridge_counts = []
    p_int = []
    cooperativity = []
    intercalated_monomer_count = []
    bridge_fraction_bound = []
    mean_bridge_count_bound = []
    times = []
    steps = []

    for col_fil, col_crow in zip(
            data.timestep[start:end:step].timestep,
            data_other.timestep[start:end:step].timestep):
        mask_stack = col_fil.particles[:].type.flatten() == 4
        mask_stack = np.arange(len(mask_stack))[mask_stack]
        stacking_sites = col_fil.particles[list(mask_stack)]
        stacking_site_ids = np.ravel(stacking_sites.id).astype(int)

        crowder_view = col_crow.select_particles_by_object(cfg.particle_group_alt, crowder_ids)
        mask_ligand = crowder_view.particles[:].type.flatten() == 5
        ligand_indices = np.arange(len(mask_ligand))[mask_ligand]
        ligands = crowder_view.particles[list(ligand_indices)]
        ligand_crowder_ids = crowder_ids[ligand_indices]

        grouped_indices = get_neighbours_cross_lattice(
            ligands.pos, stacking_sites.pos, cfg.box_dim[0], cfg.crit)
        intercalated_local_indices = [
            int(idx) for idx, neighbours in grouped_indices.items()
            if len(neighbours) == 2
        ]
        intercalated_crowder_ids = ligand_crowder_ids[intercalated_local_indices]

        per_parent = dict.fromkeys(parent_ids.tolist(), 0)
        targets_per_parent = {int(parent_id): set() for parent_id in parent_ids}
        for local_index in intercalated_local_indices:
            crowder_id = int(ligand_crowder_ids[local_index])
            parent_id = child_to_parent.get(crowder_id)
            if parent_id is None:
                continue
            per_parent[parent_id] += 1
            target_multimers = [
                stacking_site_to_multimer.get(int(stacking_site_ids[stacking_index]))
                for stacking_index in grouped_indices[local_index]
            ]
            targets_per_parent[parent_id].update(
                target_id for target_id in target_multimers if target_id is not None)

        n_int = np.array([per_parent[int(parent_id)] for parent_id in parent_ids], dtype=int)
        bridge_count = np.array([
            len(targets_per_parent[int(parent_id)]) if per_parent[int(parent_id)] else 0
            for parent_id in parent_ids
        ], dtype=int)
        counts = np.bincount(n_int, minlength=ligand_length + 1)[:ligand_length + 1]
        b_counts = np.bincount(
            bridge_count, minlength=max_bridge_count + 1)[:max_bridge_count + 1]
        joint_counts = np.zeros((ligand_length + 1, max_bridge_count + 1), dtype=int)
        np.add.at(joint_counts, (n_int, bridge_count), 1)
        occupancy_counts.append(counts)
        bridge_counts.append(b_counts)
        joint_occupancy_bridge_counts.append(joint_counts)

        p_val = float(np.mean(n_int) / ligand_length) if n_ligands else np.nan
        if ligand_length <= 1 or not np.isfinite(p_val) or p_val == 0.0:
            c_val = np.nan
        else:
            numerator = float(np.mean(n_int * (n_int - 1)))
            denominator = float(ligand_length * (ligand_length - 1) * p_val**2)
            c_val = numerator / denominator

        bound_mask = n_int > 0
        p_int.append(p_val)
        cooperativity.append(c_val)
        intercalated_monomer_count.append(int(np.sum(n_int)))
        bridge_fraction_bound.append(
            float(np.mean(bridge_count[bound_mask] >= 2)) if np.any(bound_mask) else np.nan)
        mean_bridge_count_bound.append(
            float(np.mean(bridge_count[bound_mask])) if np.any(bound_mask) else np.nan)
        times.append(float(np.ravel(col_fil.time)[0]))
        steps.append(int(np.ravel(col_fil.step)[0]))

    occupancy_counts = np.asarray(occupancy_counts, dtype=int)
    bridge_counts = np.asarray(bridge_counts, dtype=int)
    joint_occupancy_bridge_counts = np.asarray(joint_occupancy_bridge_counts, dtype=int)
    occupancy_prob = (
        occupancy_counts / float(n_ligands)
        if n_ligands else occupancy_counts.astype(float)
    )
    bridge_prob = (
        bridge_counts / float(n_ligands)
        if n_ligands else bridge_counts.astype(float)
    )
    joint_occupancy_bridge_prob = (
        joint_occupancy_bridge_counts / float(n_ligands)
        if n_ligands else joint_occupancy_bridge_counts.astype(float)
    )

    data_with_context[cfg.data_path] = {
        'N': int(ligand_length),
        'n_ligands': int(n_ligands),
        'n_values': np.arange(ligand_length + 1, dtype=int),
        'bridge_values': np.arange(max_bridge_count + 1, dtype=int),
        'time': np.asarray(times, dtype=float),
        'step': np.asarray(steps, dtype=int),
        'occupancy_counts': occupancy_counts,
        'occupancy_prob': occupancy_prob,
        'bridge_counts': bridge_counts,
        'bridge_prob': bridge_prob,
        'joint_occupancy_bridge_counts': joint_occupancy_bridge_counts,
        'joint_occupancy_bridge_prob': joint_occupancy_bridge_prob,
        'p_int': np.asarray(p_int, dtype=float),
        'C': np.asarray(cooperativity, dtype=float),
        'intercalated_monomer_count': np.asarray(intercalated_monomer_count, dtype=int),
        'bridge_fraction_bound': np.asarray(bridge_fraction_bound, dtype=float),
        'mean_bridge_count_bound': np.asarray(mean_bridge_count_bound, dtype=float),
    }
    return data_with_context

def exclude_patches(subset):
    return subset.type != 4

def is_g4_filament(subset):
    return not np.any(subset.type == 5)

kernel_kwargs={'particle_group':'Filament','particle_group_alt':'Crowder','chunk': (-5, None, 1), 'box_dim': np.array([445.15987681038695]*3), 'crit': 1.3, 'extra_flag': 'concentration_variation', 'particle_predicate': exclude_patches,'object_predicate': is_g4_filament}
engine_inst=Engine(world_path='{path_to_data}/')
engine_inst.assemble_paths(master_dict,template_hndl,'which_sim')
engine_inst.register_kernel(calculate_ligand_cooperativity, **kernel_kwargs)
engine_inst.run(max_workers=12)
engine_inst.collect_results()
engine_inst.save_results(filename='g4polyligand_coop.p.gz', custom_full_path='{path_to_data}/test_local_analysis/g4polyligand_coop.p.gz')
engine_inst.shutdown()
print('DONE!')

In [ ]:
processed_data_paths = '{path_to_data}/test_local_analysis/g4polyligand_coop.p.gz'
clr = plt.cm.tab10(np.linspace(0, 1, 4, endpoint=True))

pool_lbl, pool_data = open_data(processed_data_paths, key='calculate_ligand_cooperativity')
coop_data = defaultdict(list)
for lbl, data_el in zip(pool_lbl, pool_data):
    eps_ligand = determine_key_val_from_filename(template_hndl, lbl, 'which_vdW_ligand')
    ligand_length = determine_key_val_from_filename(template_hndl, lbl, 'what_monomer_number_ligand')
    for run_result in data_el:
        coop_data[f'{eps_ligand}_{ligand_length}'].append(list(run_result.values())[0])

plot_labels = {3: 'polyligand (N=3)', 5: 'polyligand (N=5)'}
eps_values = [10.0, 15.0, 20.0]
eps_color_index = {5.0: 0, 10.0: 1, 15.0: 2, 20.0: 3}
active_occ = defaultdict(dict)
summary_rows = []
for lbl, run_results in coop_data.items():
    eps_ligand = float(lbl.split('_')[0])
    ligand_length = int(float(lbl.split('_')[-1]))
    if eps_ligand not in eps_values or ligand_length not in plot_labels:
        continue

    occ = np.stack([res['occupancy_prob'][-1] for res in run_results])
    p_from_kernel = np.array([res['p_int'][-1] for res in run_results])
    n_values = run_results[0]['n_values']
    p_from_pn = np.sum(occ * n_values, axis=1) / ligand_length
    occ_mean = np.mean(occ, axis=0)
    p_bound = 1.0 - occ_mean[0]
    active_prob = occ_mean[1:] / p_bound if p_bound > 0 else np.full(ligand_length, np.nan)
    active_occ[eps_ligand][ligand_length] = active_prob

    summary_rows.append((
        eps_ligand,
        ligand_length,
        np.mean(p_from_kernel),
        np.mean(p_from_pn),
        np.max(np.abs(p_from_kernel - p_from_pn)),
        active_prob,
    ))

for eps_ligand in eps_values:
    fig_occ, ax_occ = plt.subplots(figsize=(3.2, 2.8), constrained_layout=True)
    color = clr[eps_color_index[eps_ligand]]

    pentamer = active_occ[eps_ligand][5]
    ax_occ.bar(
        np.arange(1, 6), pentamer,
        width=0.72, color=color, alpha=0.38,
        edgecolor=color, linewidth=1.0,
        label=plot_labels[5],
        zorder=1,
    )

    trimer = active_occ[eps_ligand][3]
    ax_occ.bar(
        np.arange(1, 4), trimer,
        width=0.34, color=color, alpha=0.95,
        edgecolor='black', linewidth=0.7,
        label=plot_labels[3],
        zorder=2,
    )

    ax_occ.set_title(rf'$\epsilon={eps_ligand:g}$')
    ax_occ.set_xlabel(r'$n_\mathrm{int}$')
    ax_occ.set_ylabel(r'$P(n_\mathrm{int})$')
    ax_occ.set_xticks(np.arange(1, 6))
    ax_occ.set_ylim(0, 1)
    ax_occ.legend(frameon=False, labelspacing=0.15, handletextpad=0.3, borderpad=0.1, fontsize=10)
    ax_occ.grid(True, axis='y', alpha=0.3)
    ax_occ.spines[['top', 'right']].set_visible(False)

In [ ]:

from pathlib import Path
import gzip
import pickle

processed_data_paths = '{path_to_data}/test_local_analysis/g4polyligand_coop.p.gz'

with gzip.open(processed_data_paths, 'rb') as fp:
    processed = pickle.load(fp)['calculate_ligand_cooperativity']

rows = []
for lbl, run_results in processed.items():
    eps_ligand = float(determine_key_val_from_filename(template_hndl, lbl, 'which_vdW_ligand'))
    ligand_length = int(float(determine_key_val_from_filename(template_hndl, lbl, 'what_monomer_number_ligand')))
    for run_result in run_results:
        path, res = next(iter(run_result.items()))
        sim = Path(path).parent.name
        Pn = res['occupancy_prob'][-1]
        joint = res['joint_occupancy_bridge_prob'][-1]
        p_bound = 1.0 - float(Pn[0])
        p_full = float(Pn[-1])
        if p_bound > 0:
            mean_n_bound_over_N = float(np.sum(Pn * np.arange(ligand_length + 1)) / p_bound / ligand_length)
            bridge_fraction_bound = float(res['bridge_fraction_bound'][-1])
            mean_bridge_count_bound = float(res['mean_bridge_count_bound'][-1])
        else:
            mean_n_bound_over_N = np.nan
            bridge_fraction_bound = np.nan
            mean_bridge_count_bound = np.nan
        rows.append({
            'sim': sim,
            'epsilon': eps_ligand,
            'N': ligand_length,
            'p_int': float(res['p_int'][-1]),
            'P_bound': p_bound,
            'mean_n_bound_over_N': mean_n_bound_over_N,
            'P_full': p_full,
            'P_partial_bound': p_bound - p_full,
            'bridge_fraction_bound': bridge_fraction_bound,
            'mean_bridge_count_bound': mean_bridge_count_bound,
            'P_partial_bridge': float(np.sum(joint[1:ligand_length, 2:])),
            'P_full_bridge': float(np.sum(joint[ligand_length, 2:])),
            'P_any_bridge': float(np.sum(joint[1:, 2:])),
        })

rows = sorted(rows, key=lambda row: (row['epsilon'], row['N'], row['sim']))

eps_values = np.array(sorted({row['epsilon'] for row in rows}))
plot_labels = {1: 'monoligand (N=1)', 3: 'polyligand (N=3)', 5: 'polyligand (N=5)'}
colors_by_N = {1: '#4C4C4C', 3: '#2C7BB6', 5: '#D95F02'}
markers_by_N = {1: 'o', 3: 's', 5: '^'}

def mean_std(field, ligand_length):
    means = []
    stds = []
    for eps in eps_values:
        values = np.array([
            row[field] for row in rows
            if row['N'] == ligand_length and row['epsilon'] == eps
        ], dtype=float)
        values = values[np.isfinite(values)]
        means.append(np.mean(values) if len(values) else np.nan)
        stds.append(np.std(values) if len(values) else np.nan)
    return np.array(means), np.array(stds)

fig_bridge, ax_bridge = plt.subplots(figsize=(4.6, 3.5), constrained_layout=True)
for ligand_length in (1, 3, 5):
    y, yerr = mean_std('bridge_fraction_bound', ligand_length)
    ax_bridge.errorbar(
        eps_values, y, yerr=yerr,
        color=colors_by_N[ligand_length], marker=markers_by_N[ligand_length],
        markerfacecolor='none', capsize=3, label=plot_labels[ligand_length],
        linewidth=1.8, markersize=6,
    )
ax_bridge.set_xlabel(r'$\epsilon$')
ax_bridge.set_ylabel(r'$P_B$')
ax_bridge.set_xticks(eps_values)
ax_bridge.set_ylim(-0.01, 0.18)
ax_bridge.legend(frameon=False, labelspacing=0.15, handletextpad=0.3, borderpad=0.1, fontsize=10)
ax_bridge.grid(True, axis='y', alpha=0.3)
ax_bridge.spines[['top', 'right']].set_visible(False)


In [ ]:
processed_data_paths = '{path_to_data}/test_local_analysis/g4polyligand_coop.p.gz'

pool_lbl, pool_data = open_data(processed_data_paths, key='calculate_ligand_cooperativity')
toc_values = {}
toc_stds = {}
for lbl, data_el in zip(pool_lbl, pool_data):
    eps_ligand = float(determine_key_val_from_filename(template_hndl, lbl, 'which_vdW_ligand'))
    ligand_length = int(float(determine_key_val_from_filename(template_hndl, lbl, 'what_monomer_number_ligand')))
    if eps_ligand != 10.0 or ligand_length not in (1, 3):
        continue
    final_values = np.array([list(run_result.values())[0]['p_int'][-1] for run_result in data_el], dtype=float)
    toc_values[ligand_length] = np.mean(final_values)
    toc_stds[ligand_length] = np.std(final_values)

fig_toc, ax_toc = plt.subplots(figsize=(2.4, 3.0), constrained_layout=True)
toc_colors = {1: '#2ca02c', 3: clr[1]}
for xpos, ligand_length, alpha in [(0, 1, 0.95), (1, 3, 0.95)]:
    ax_toc.bar(
        xpos,
        100 * toc_values[ligand_length],
        color=toc_colors[ligand_length],
        alpha=alpha,
        edgecolor='black',
        linewidth=0.7,
    )
ax_toc.set_xticks([0, 1])
ax_toc.set_xticklabels(['monoligand', 'polyligand'], rotation=0, ha='center')
ax_toc.set_ylabel('Probability [%]')
ax_toc.set_ylim(0, 100 * max(toc_values.values()) * 1.35)
ax_toc.grid(True, axis='y', alpha=0.28)
ax_toc.spines[['top', 'right']].set_visible(False)
for xpos, ligand_length in [(0, 1), (1, 3)]:
    ax_toc.text(
        xpos,
        100 * toc_values[ligand_length] + 100 * max(toc_values.values()) * 0.05,
        f'{100 * toc_values[ligand_length]:.1f}',
        ha='center',
        va='bottom',
        fontsize=10,
    )

In [ ]:
from pmtools.runner import Engine
from pmtools.kernels import Ree_segments
import numpy as np

def exclude_patches(subset):
    return subset.type != 4

def is_g4_filament(subset):
    return not np.any(subset.type == 5)

kernel_kwargs={'particle_group':'Filament','particle_group_alt':'Crowder','chunk': (-5, None, 1), 'box_dim': np.array([445.15987681038695]*3), 'crit': 1.3, 'extra_flag': 'concentration_variation', 'particle_predicate': exclude_patches,'object_predicate': is_g4_filament}

engine_inst = Engine(world_path='{path_to_data}/')
engine_inst.assemble_paths(master_dict, template_hndl, 'which_sim')
engine_inst.register_kernel(Ree_segments, **kernel_kwargs)
engine_inst.run(max_workers=12)
engine_inst.collect_results()
engine_inst.save_results(filename='g4polyligand_ree_segments.p.gz',
    custom_full_path='{path_to_data}/test_local_analysis/g4polyligand_ree_segments.p.gz')
engine_inst.shutdown()
print('DONE!')


In [ ]:
from pmtools.refractored_toolbox import determine_key_val_from_filename

processed_data_path = '{path_to_data}/test_local_analysis/g4polyligand_ree_segments.p.gz'
pool_lbl, pool_data = open_data(processed_data_path, key='Ree_segments')

res_dict_Ree = defaultdict(list)
segment_means = defaultdict(list)

for lbl, run_results in zip(pool_lbl, pool_data):
    epsilon = determine_key_val_from_filename(
        template_hndl, lbl, 'which_vdW_ligand'
    )
    ligand_length = determine_key_val_from_filename(
        template_hndl, lbl, 'what_monomer_number_ligand'
    )
    condition = f'{epsilon}_{ligand_length}'

    for run_result in run_results:
        ree_values, segment_values = next(iter(run_result.values()))
        res_dict_Ree[condition].extend(ree_values)
        segment_means[condition].append(
            np.mean(np.asarray(segment_values, dtype=float), axis=0)
        )

res_dict_Ree = {
    condition: np.abs(np.asarray(values, dtype=float))
    for condition, values in res_dict_Ree.items()
}
res_dict_contour = {
    condition: np.sum(np.mean(np.asarray(values, dtype=float), axis=0))
    for condition, values in segment_means.items()
}

for condition in sorted(res_dict_Ree):
    print(
        condition,
        f'R_ee = {np.mean(res_dict_Ree[condition]):.4f}',
        f'L_c = {res_dict_contour[condition]:.4f}',
        f'n = {len(res_dict_Ree[condition])}',
    )


In [ ]:
N = 15
sigma = 8.5
key_f = '5.0_1.0'
R_f = np.mean(res_dict_Ree[key_f])
R_f_sq_mean = np.mean(res_dict_Ree[key_f] ** 2)
a_f = R_f_sq_mean / res_dict_contour[key_f]
v = np.pi / 6.0 * sigma ** 3


def calc_A(R_fl, a_fl):
    return (
        R_f ** 2 / (N * a_f ** 2)
        - R_fl ** 2 / (N * a_fl ** 2)
        + v * N ** 2 * (1 / R_f ** 3 - 1 / R_fl ** 3)
    )


print(f'Reference: R_f = {R_f:.4f}, a_f = {a_f:.4f}')
for key in sorted(res_dict_Ree):
    R_fl = np.mean(res_dict_Ree[key])
    a_fl = R_f_sq_mean / res_dict_contour[key]
    print(key, R_fl, a_fl, calc_A(R_fl, a_fl))


In [ ]:
from pathlib import Path

# A mapped over delta-R and delta-a (relative to reference), with simulated points.
R_ref = R_f
a_ref = a_f
R_ref_sq_mean = R_f_sq_mean
output_dir = Path('figures')
output_dir.mkdir(exist_ok=True)
figure_path = output_dir / 'melting_temp_final.pdf'
csv_export_path = output_dir / 'melting_temp_final_data.csv'
export_rows = []

sim_points = []
for key in sorted(res_dict_Ree):
    R_mean = np.mean(res_dict_Ree[key])
    a_mean = R_ref_sq_mean / res_dict_contour[key]
    dR = R_mean - R_ref
    da = a_mean - a_ref
    sim_points.append((key, dR, da, calc_A(R_mean, a_mean), R_mean, a_mean))

dR_grid = np.linspace(-5, 25, 140)
da_grid = np.linspace(-2, 4.5, 140)
dR_mesh, da_mesh = np.meshgrid(dR_grid, da_grid)

R_mesh = R_ref + dR_mesh
a_mesh = a_ref + da_mesh
A_mesh = (
    R_ref ** 2 / (N * a_ref ** 2)
    - R_mesh ** 2 / (N * a_mesh ** 2)
    + v * N ** 2 * (1 / R_ref ** 3 - 1 / R_mesh ** 3)
)

for key, dR, da, A_val, R_mean, a_mean in sim_points:
    eps_str, m_str = key.split('_')
    eps_i, m_i = int(float(eps_str)), int(float(m_str))
    label = rf'({eps_i}, {m_i})'
    export_rows.append({
        'row_type': 'sampled_point',
        'epsilon': eps_i,
        'M_ligand_monomer_count': m_i,
        'figure_point_label': label,
        'delta_R_FL_minus_R_F': float(dR),
        'delta_a_FL_minus_a_F': float(da),
        'A_equation_value': float(A_val),
        'R_FL_mean': float(R_mean),
        'a_FL_mean': float(a_mean),
        'R_F_reference': float(R_ref),
        'a_F_reference': float(a_ref),
        'R_F_squared_mean': float(R_ref_sq_mean),
        'N_backbone_monomer_count': float(N),
        'v_interaction_parameter': float(v),
    })

plt.figure(figsize=(6.5, 5.5))
cp = plt.contourf(dR_mesh, da_mesh, A_mesh, levels=25, cmap='coolwarm')
cb = plt.colorbar(cp)
cb.set_label('A', rotation=270, labelpad=12)
plt.contour(
    dR_mesh, da_mesh, A_mesh,
    levels=[0], colors='k', linewidths=1, linestyles='--',
)

annotation_offsets = {
    '(15, 5)': (0, -6, 'left', 'top'),
    '(20, 5)': (0, -6, 'left', 'top'),
    '(5, 1)': (8, 6, 'left', 'top'),
    '(5, 3)': (-4, -6, 'right', 'top'),
    '(10, 1)': (3, 5, 'left', 'bottom'),
    '(5, 5)': (-5, 5, 'right', 'bottom'),
}
for key, dR, da, A_val, R_mean, a_mean in sim_points:
    eps_str, m_str = key.split('_')
    label = rf'({int(float(eps_str))}, {int(float(m_str))})'
    plt.scatter(dR, da, c='k', s=40, edgecolor='white', zorder=3)
    dx, dy, ha, va = annotation_offsets.get(label, (0, 5, 'right', 'bottom'))
    plt.annotate(
        label, (dR, da), fontsize=10,
        textcoords='offset points', xytext=(dx, dy), ha=ha, va=va,
    )

plt.xlabel(r'$R_{FL}-R_F$')
plt.ylabel(r'$a_{FL}-a_F$')
plt.grid(linestyle='dotted', alpha=0.5)
plt.tight_layout()
